# 🧠 NEXT-WORD PREDICTION

## PROJECT OVERVIEW

This project builds a next-word prediction and text-generation system using NLP and Deep Learning.

The project includes Text Cleaning, Tokenization, Sequence Creation, Padding, One-Hot Encoding, and Model Training to learn patterns between words and predict the next word.

Two neural network models were explored: Simple RNN and LSTM. The LSTM model was used for the final text generation because it can better retain information from earlier words.

The LSTM model was trained using EarlyStopping to reduce unnecessary training and help prevent overfitting.

**Model Used: LSTM**

Techniques: Text Cleaning, Tokenization, Padding, One-Hot Encoding, Next-Word Prediction

Evaluation/Training: Training & Validation Loss, Training & Validation Accuracy

Final Result: Predicts the next word and generates longer text from a given starting phrase.

**Import Libraries**

In [51]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

**Load Data**

In [52]:
df = pd.read_csv("qoute_dataset.csv")

In [53]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [54]:
df.shape

(3038, 2)

# Data Understanding and Preprocessing

In [55]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


**Separate Qoutes and Authors**

**Convert into lowercase**

In [56]:
# Convert all text in the 'quotes' column to lowercase
quotes = quotes.str.lower()

**Remove Punctuation**

In [57]:
import string  # Import Python's built-in string module

# Create a translator that removes all punctuation characters
translator = str.maketrans('', '', string.punctuation)

# Apply the translator to each value in the 'quotes' column
# This removes punctuation such as .,!?;:"' from the text
quotes = quotes.apply(lambda x: x.translate(translator))

In [58]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


# Tokenizer

## Create Token

In [59]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Set the maximum number of words in the vocabulary
vocab_size = 10000

# Create a tokenizer with the specified vocabulary size
tokinizer = Tokenizer(num_words=vocab_size)

# Learn the vocabulary from the quotes
tokinizer.fit_on_texts(quotes)

# Get the word-to-index mapping
word_index = tokinizer.word_index

# Print the total number of unique words found in the quotes
print(len(word_index))

# Display the first 10 word-to-index pairs
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

We created a tokenizer that learned the vocabulary from the quotes and assigned a unique number to each word.

## Create Sequence

In [60]:
# Convert each quote into a sequence of numbers using the tokenizer
sequence = tokinizer.texts_to_sequences(quotes)

# Print the first 3 original quotes
for i in range(3):
    print(quotes[i])

# Print the corresponding numerical sequences for the first 3 quotes
for i in range(3):
    print(sequence[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”
[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


We converted the text quotes into numerical sequences, where each word is represented by its assigned number.

## Create Input-Output Sequences for Next-Word Prediction

In [61]:
# Create empty lists to store input and output sequences
X = []
y = []

# Loop through each numerical sequence
for seq in sequence:

    # Create multiple input-output pairs from each sequence
    for i in range(1, len(seq)):

        # Use the previous words as the input sequence
        input_seq = seq[:i]

        # Use the next word as the output
        output_seq = seq[i]

        # Add the input sequence to X
        X.append(input_seq)

        # Add the output word to y
        y.append(output_seq)

We created training samples where previous words (X) are used to predict the next word (y). This prepares the text data for next-word prediction.

For example, if the quote is:

**"life is what happens when"**

We create training pairs like:

Input → Output

life → is

life is → what

life is what → happens

life is what happens → when

So:

X = previous words → what the model sees
y = next word → what the model should predict

This teaches the neural network how words follow each other, so later it can predict the next word when you give it a sentence.

## Pad Input Sequences for Model Training

In [62]:
# Check the total number of input sequences
len(X)

85271

In [63]:
# Check the total number of output words
len(y)

85271

In [64]:
# Find the length of the longest input sequence
max_len = max(len(x) for x in X)

# Print the maximum sequence length
print(max_len)

from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pad shorter sequences with zeros at the beginning
# so that all input sequences have the same length
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

# Convert the output labels into a NumPy array
y = np.array(y)

# Display the shape of the padded input data
X_padded.shape

745


(85271, 745)

We made all input sequences the same length by adding zeros to shorter sequences. This makes the data ready to be used as input for a neural network.

## Convert Output Labels to One-Hot Encoding

In [65]:
from tensorflow.keras.utils import to_categorical

# Convert each output word into a one-hot encoded vector
y_one_hot = to_categorical(y, num_classes=vocab_size)

# Check the shape of the original output labels
y.shape

(85271,)

In [66]:
# Check the shape of the one-hot encoded output labels
y_one_hot.shape

(85271, 10000)

We converted each output word into a one-hot encoded vector, so the model can learn to predict one word from the vocabulary as the next word.

# Model Creation

## Build the Simple RNN Model

In [67]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

# Set the size of the word embeddings
embedding_dim = 50

# Set the number of units in the RNN layer
rnn_units = 128

# Create a sequential neural network model
rnn_model = Sequential()

# Convert word indices into dense vector representations
rnn_model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_len
    )
)

# Add a SimpleRNN layer to learn patterns between words
rnn_model.add(SimpleRNN(units=rnn_units))

# Add an output layer to predict the next word
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

**Compile the Simple RNN Model**

In [68]:
# Configure the model for training
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

We created a Simple RNN model for next-word prediction. The model takes previous words as input, learns the patterns between them, and predicts what word should come next.

The model summary shows the three layers we created:

Embedding → SimpleRNN → Dense

Embedding: Converts word numbers into useful numerical vectors.

SimpleRNN: Learns the relationship and sequence of words.

Dense + Softmax: Chooses the most likely next word from the vocabulary.

Compile: Sets the optimizer and loss function used during training.

0 (unbuilt) means the model has not received actual input data yet, so Keras has not calculated the output shapes and number of parameters.

In [69]:
rnn_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

The summary shows the layers, output shapes, and parameters of our Simple RNN model. It helps us understand and check the model structure before training.

## Build the LSTM Model

In [70]:
# Create a new sequential model
lstm_model = Sequential()

# Convert word numbers into meaningful vector representations
lstm_model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=max_len
    )
)

# Add an LSTM layer to learn patterns and relationships between words
lstm_model.add(LSTM(units=rnn_units))

# Add an output layer to predict the next word
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

**Compile the LSTM Model**

In [71]:
# Configure the model for training
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

We created an LSTM model for next-word prediction, similar to the Simple RNN model.

Model structure: Embedding → LSTM → Dense

The main difference is that LSTM can remember important information from earlier words better, especially in longer sequences. The final Dense + Softmax layer predicts the most likely next word.

In [72]:
# Display the structure and details of the LSTM model
lstm_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

The model summary shows the layers, output shapes, and number of parameters in our LSTM model.

Model structure: Embedding → LSTM → Dense

It helps us check how our model is built before training.

# Train the Model

We only train the LSTM model for the final result because it can remember important information from earlier words better than a Simple RNN, especially for longer sequences. Therefore, we used LSTM for our final next-word prediction and text generation.

## Train the Simple LSTM Model


**Use EarlyStopping**

In [73]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop training when validation loss stops improving
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

EarlyStopping is useful so the LSTM doesn't keep training when validation performance stops improving.

We added EarlyStopping to stop training when the validation loss does not improve for 5 epochs.

restore_best_weights=True keeps the best-performing model weights, so the model doesn't end with a worse version after extra training.

In [ ]:
# Set the maximum number of training epochs
epochs = 100

# Set the number of samples processed at one time
batch_size = 128

# Train the LSTM model with early stopping
history_lstm = lstm_model.fit(
    X_padded,
    y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    callbacks=[early_stopping]
)

We trained the LSTM model using our input sequences and target words.

100 epochs: The model can learn from the training data up to 100 times.

Batch size = 128: The model processes 128 samples at a time.

Validation split = 0.1: 10% of the data is used to check the model during training.

EarlyStopping: Stops training if the validation loss does not improve for 5 epochs and restores the best model weights.

history_lstm: Stores the training and validation accuracy and loss so we can analyze the model's performance later.

**Save the Model**

For Saving the Model use this code:

`lstm_model.save(lstm_model.h5)`

# Testing

**Load the Model**

In [81]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm_model.h5")

## Create Index-to-Word Mapping

In [82]:
# Create an empty dictionary to map numbers back to words
index_to_word = {}

# Loop through each word and its assigned index
for word, index in word_index.items():

    # Store the word using its index as the key
    index_to_word[index] = word

We created a reverse mapping from word numbers back to their original words.

For example:

word_index:     "love" → 25

index_to_word:  25 → "love"

This allows us to convert the model's predicted number back into a word.

## Create a Next-Word Predictor Function

In [83]:
# Create a function to predict the next word
def predictor(model, tokenizer, text, max_len):

    # Convert the input text to lowercase
    text = text.lower()

    # Convert the text into a sequence of word numbers
    seq = tokenizer.texts_to_sequences([text])[0]

    # Make the sequence the same length as the model's input
    seq = pad_sequences([seq], maxlen=max_len, padding='pre')

    # Use the model to predict the next word
    pred = model.predict(seq, verbose=0)

    # Find the index of the word with the highest probability
    pred_index = np.argmax(pred)

    # Convert the predicted index back into a word
    return index_to_word[pred_index]

**Give Text to predict next Word**

In [84]:
# Define the starting text
seed_text = "what are you"

# Predict the next word using the trained LSTM model
next_word = predictor(lstm_model, tokinizer, seed_text, max_len)

# Print the predicted next word
print(next_word)

worrying


We created a prediction function that takes a starting sentence, converts it into numbers, and gives it to our trained LSTM model. The model then predicts the most likely next word and converts the predicted number back into a word.

## Generate Multiple Words Using the LSTM Model

In [85]:
# Create a function to generate multiple words from a starting sentence
def generate_text(model, tokenizer, seed_text, max_len, n_words):

    # Generate one word at a time
    for _ in range(n_words):

        # Predict the next word using the trained model
        next_word = predictor(model, tokenizer, seed_text, max_len)

        # Stop if no word is predicted
        if next_word == "":
            break

        # Add the predicted word to the existing sentence
        seed_text += " " + next_word

    # Return the complete generated sentence
    return seed_text

**Give Text to predict next Words**

In [87]:
# Define the starting sentence
seed = "are you a"

# Generate 10 new words using the trained LSTM model
generated_text = generate_text(
    lstm_model,
    tokinizer,
    seed,
    max_len,
    10
)

# Print the generated text
print(generated_text)

are you a thousand times i wrote the less if it does not


We created a text generation function that repeatedly predicts the next word and adds it to the sentence. This allows the LSTM model to generate a longer sentence from a starting word or phrase.

**Save the Tokenizer and Maximum Sequence Length**

In [88]:
import pickle

# Save the trained tokenizer to a file
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokinizer, f)

# Save the maximum sequence length to a file
with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)

We saved the tokenizer and maximum sequence length as .pkl files so we can reuse them later when loading the model and making predictions without training the tokenizer again.

# Conclusion

We built a next-word prediction system using NLP and Deep Learning. We cleaned and tokenized the text, converted words into numerical sequences, padded the sequences, and applied one-hot encoding. We built and compared Simple RNN and LSTM models, then trained the LSTM with 100 epochs, batch size 128, 10% validation data, and EarlyStopping. Finally, we created functions to predict the next word and generate longer text, and saved the tokenizer and sequence length for future use.

**Why We Used LSTM Instead of Simple RNN**

We created the Simple RNN to understand the basic idea of recurrent neural networks and next-word prediction. However, we used LSTM for the final text generation because LSTM has a memory mechanism that can better retain important information from earlier words, especially when sequences become longer.

**Final Achievement**

We successfully created an LSTM-based next-word prediction and text-generation system that takes a starting phrase, predicts the next word, and continues generating words based on patterns learned from the training data.